# Notebook 01: Data Pipeline

Builds the three processed CSV files that all downstream notebooks depend on:

- `data/abagym_antibody.csv` -- 5318 mutation rows with reconstructed sequences and CDR/FR region labels
- `data/abagym_sequences.csv` -- 5 rows (one per antibody) with wildtype sequences and ANARCI position mappings
- `data/sabdab_affinity.csv` -- 491 antibody-antigen pairs with cleaned sequences and pKd labels

CSVs are saved to `data/` in the repo (git-tracked), not to Drive. This makes them available after a simple `git clone` without needing Drive mounted. Drive is reserved for large files (embeddings, checkpoints).

**Local setup:** Skip cells 1-7 (Colab only). Start from the Imports cell. HMMER must be installed via `brew install hmmer` before running ANARCI mapping.

**Colab setup:** Run all cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys

REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
REPO_DIR = '/content/antibody-property-prediction'
BRANCH = 'implementation'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project/Antibody_Project')
# DATA_DIR is in the repo (data/ at repo root) -- comes from src.config
EMBEDDING_DIR = DRIVE_ROOT / 'embeddings'
RESULTS_DIR = DRIVE_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set.")

In [ ]:
!apt-get install -y hmmer
!pip install -q fair-esm ablang2 anarci wandb

In [ ]:
!pip install -q --upgrade ipython

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
subprocess.run(['find', '/content/antibody-property-prediction', '-type', 'd',
                '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
               capture_output=True)
print("Autoreload enabled, pycache cleared.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import wandb
wandb.login()

## Imports

Pipeline functions from `src/data/abagym.py` and `src/data/sabdab.py` handle all data logic. This notebook is orchestration only -- no data processing logic lives here.

In [1]:
import re
import json
import subprocess

import pandas as pd
from pathlib import Path

from src.config import DATA_DIR, DRIVE_ROOT, ABAGYM_DATASETS, N_MUTATIONS, N_SABDAB
from src.data.abagym import (
    load_abagym_antibody,
    load_abagym_sequences,
    PDB_FILE_STEMS,
    extract_chain_residues,
    build_anarci_mapping,
    reconstruct_mutant_sequence,
)
from src.data.sabdab import load_sabdab, parse_sabdab_raw, SABDAB_URL

## AbAgym Data

AbAgym (Antibody-Antigen Gym) is a benchmark of deep mutational scanning (DMS) experiments. Each experiment measures the effect of every possible single amino acid substitution at a set of positions on antibody binding. We use 5 of the 68 available datasets, selected to cover a range of CDR/FR mutation ratios and antibody scaffolds.

The repo is cloned to Drive and cached across sessions. It contains zip files -- PDB structures and DMS data are extracted on first run.

After cloning, this section:
1. Extracts and verifies the 5 PDB files for our antibodies
2. Loads the combined DMS CSV, inspects its structure, and filters to our 5 datasets
3. Runs ANARCI on each PDB chain to build position mappings (PDB label → IMGT number → CDR/FR region → seq_idx)

In [4]:
# Clone the AbAgym repo to Drive (cached on re-run)
ABAGYM_CLONE_DIR = DRIVE_ROOT / 'AbAgym_repo'

if not ABAGYM_CLONE_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/3BioCompBio/AbAgym.git',
         str(ABAGYM_CLONE_DIR)],
        check=True,
    )
    print(f"Cloned to {ABAGYM_CLONE_DIR}")
else:
    print(f"Using cached clone at {ABAGYM_CLONE_DIR}")

print("\nRepo contents:")
for p in sorted(ABAGYM_CLONE_DIR.iterdir()):
    print(f"  {p.name}")

Using cached clone at /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/AbAgym_repo

Repo contents:
  .git
  AbAgym_data_full.csv.zip
  AbAgym_data_full_interface.csv
  AbAgym_data_non-redundant.csv.zip
  AbAgym_data_non-redundant_interface.csv
  AbAgym_metadata.csv
  DMS_big_table_PDB_files
  PDB_files.zip
  README.md


### AbAgym repo file descriptions

| File | Rows | Notes |
|---|---|---|
| `AbAgym_data_full.csv.zip` | 572,719 | All 68 DMS experiments, all mutations. Columns: `chain`, `mut_name` (singular) |
| `AbAgym_data_non-redundant.csv.zip` | 323,752 | Redundant experiments removed at dataset level. Columns: `chains`, `mut_names` (match our schema) |
| `AbAgym_data_full_interface.csv` | 37,259 | Full dataset, interface-adjacent residues only |
| `AbAgym_data_non-redundant_interface.csv` | 36,541 | Non-redundant, interface residues only |
| `AbAgym_metadata.csv` | 68 | One row per experiment: antigen, PDB ID, publication DOI |
| `PDB_files.zip` | 68 files | Processed PDB structures, one per experiment. Chains renamed to H and L. Extracts to `DMS_big_table_PDB_files/`. |

**Why non-redundant:** Our 5 datasets have identical row counts (5318) in both zip files, so the redundancy filter removed nothing from our subset. The non-redundant CSV's column names already match our schema, so no renaming is needed.

**Why not the interface-filtered CSVs:** Interface-adjacent residues are those within a short distance of the antibody-antigen binding interface (measured by `closest_interface_atom_distance`). We use the full CSV to preserve the complete CDR/FR signal -- framework mutations far from the interface are the primary source of the CDR constraint gradient.

In [5]:
import zipfile

# Extract PDB files -- zip extracts to DMS_big_table_PDB_files/
pdb_dir = ABAGYM_CLONE_DIR / 'DMS_big_table_PDB_files'
if not pdb_dir.exists():
    with zipfile.ZipFile(ABAGYM_CLONE_DIR / 'PDB_files.zip') as zf:
        zf.extractall(ABAGYM_CLONE_DIR)
    print("Extracted PDB_files.zip")
else:
    print("PDB files already extracted")

print(f"\nChecking our 5 PDB files in {pdb_dir.name}/:")
for dms_name, stem in PDB_FILE_STEMS.items():
    pdb_path = pdb_dir / f'{stem}.pdb'
    status = "OK" if pdb_path.exists() else "MISSING"
    print(f"  [{status}] {stem}.pdb")

PDB files already extracted

Checking our 5 PDB files in DMS_big_table_PDB_files/:
  [OK] G6_27_30A_corrected_4zfg.pdb
  [OK] Cetuximab_1yy9.pdb
  [OK] D441_1mlc.pdb
  [OK] G6_27_30A_corrected_4zff.pdb
  [OK] trastuzumab_8pwh.pdb


In [6]:
# Extract and inspect the non-redundant DMS CSV
data_zip = ABAGYM_CLONE_DIR / 'AbAgym_data_non-redundant.csv.zip'
with zipfile.ZipFile(data_zip) as zf:
    csv_name = zf.namelist()[0]
    data_csv_path = ABAGYM_CLONE_DIR / csv_name
    if not data_csv_path.exists():
        zf.extractall(ABAGYM_CLONE_DIR)
        print(f"Extracted {csv_name}")
    else:
        print(f"Using cached {csv_name}")

raw_df = pd.read_csv(data_csv_path, dtype={'site': str})
print(f"Shape: {raw_df.shape}")
print(f"Columns: {raw_df.columns.tolist()}")
print(f"\nAll DMS_names ({raw_df['DMS_name'].nunique()} total):")
print(sorted(raw_df['DMS_name'].unique()))
print()
print(raw_df.head(3))

Extracted AbAgym_data_non-redundant.csv
Shape: (323752, 11)
Columns: ['DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation', 'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score', 'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance']

All DMS_names (68 total):
['Ang2_2017_G6', 'COVID-19_2021a_AZD1061', 'COVID-19_2021a_AZD8895', 'COVID-19_2021c_C002', 'COVID-19_2021c_C105', 'COVID-19_2021c_C110', 'COVID-19_2021c_C135', 'COVID-19_2021c_C144', 'COVID-19_2021c_CR3022', 'COVID-19_2021c_LY-CoV016', 'COVID-19_2021c_LY-CoV555', 'COVID-19_2021c_REGN10933', 'COVID-19_2021c_REGN10987', 'COVID-19_2021d_S2D106', 'COVID-19_2021d_S2E12', 'COVID-19_2021d_S2H13', 'COVID-19_2021d_S2H14', 'COVID-19_2021d_S2H97', 'COVID-19_2021d_S2X259', 'COVID-19_2021d_S2X35', 'COVID-19_2021d_S304', 'COVID-19_2021d_S309', 'COVID-19_2022_BD55-5840', 'COVID-19_2022_C119', 'COVID-19_2022_C121', 'COVID-19_2022_COV2-2130', 'COVID-19_2022_COV2-2196', 'COVID-19_2022_COVA2-04', 'COVID-19_2022_LY-C

In [7]:
# Filter to our 5 datasets -- columns already match our schema
combined_df = raw_df[raw_df['DMS_name'].isin(ABAGYM_DATASETS)].copy().reset_index(drop=True)

print(f"Rows: {len(combined_df)} (expected {N_MUTATIONS})")
print(f"Datasets present: {sorted(combined_df['DMS_name'].unique())}")
print(f"Columns: {combined_df.columns.tolist()}")
print()
print(combined_df[['DMS_name', 'chains', 'site', 'wildtype', 'mutation', 'mut_names']].head())

Rows: 5318 (expected 5318)
Datasets present: ['Ang2_2017_G6', 'EGFR_2013_Cetuximab', 'HER2_2021_trastuzumab', 'VEGF_2017b_G6', 'lysozyme_2019_D441']
Columns: ['DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation', 'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score', 'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance']

       DMS_name chains site wildtype mutation mut_names
0  Ang2_2017_G6      H  100        P        A    PH100A
1  Ang2_2017_G6      H  100        P        C    PH100C
2  Ang2_2017_G6      H  100        P        D    PH100D
3  Ang2_2017_G6      H  100        P        E    PH100E
4  Ang2_2017_G6      H  100        P        F    PH100F


In [8]:
# Build ANARCI mappings and wildtype sequences for all 5 antibodies
wt_sequences    = {}   # dms_name -> {'H': str, 'L': str}
anarci_mappings = {}   # dms_name -> {'H': mapping_dict, 'L': mapping_dict}

for dms_name in ABAGYM_DATASETS:
    stem     = PDB_FILE_STEMS[dms_name]
    pdb_path = pdb_dir / f'{stem}.pdb'

    h_residues = extract_chain_residues(str(pdb_path), 'H')
    l_residues = extract_chain_residues(str(pdb_path), 'L')

    h_map, _, _ = build_anarci_mapping(dms_name, 'H', h_residues)
    l_map, _, _ = build_anarci_mapping(dms_name, 'L', l_residues)

    wt_sequences[dms_name]    = {
        'H': ''.join(aa for _, aa in h_residues),
        'L': ''.join(aa for _, aa in l_residues),
    }
    anarci_mappings[dms_name] = {'H': h_map, 'L': l_map}
    print(f"  {dms_name}: H={len(h_residues)} res, L={len(l_residues)} res")

print("\nDone. Spot-check Ang2_2017_G6 chain H site 100A:")
sample = anarci_mappings['Ang2_2017_G6']['H'].get('100A')
print(f"  {sample}")  # expect imgt_pos=113, region=CDR_H3

  Ang2_2017_G6: H=215 res, L=213 res
  EGFR_2013_Cetuximab: H=220 res, L=211 res
  lysozyme_2019_D441: H=218 res, L=214 res
  VEGF_2017b_G6: H=211 res, L=213 res
  HER2_2021_trastuzumab: H=220 res, L=214 res

Done. Spot-check Ang2_2017_G6 chain H site 100A:
  {'imgt_pos': 113, 'imgt_ins': '', 'region': 'CDR_H3', 'seq_idx': 104}


## SAbDab Download

SAbDab (Structural Antibody Database) provides experimentally measured antibody-antigen binding affinities. We use a curated subset from Zenodo (DOI 10.5281/zenodo.13120765, Apache 2.0).

The raw CSV has 493 entries with raw Kd values (`Y`, in molar units). Cleaning steps applied by `parse_sabdab_raw`:
- Parse the `Antibody` column (stored as a Python list literal) into separate `heavy_seq` and `light_seq` strings
- Compute `pKd = -log10(Y)` -- the training label. Higher pKd = stronger binding (lower Kd). Mean ~8.23, range 3.70-12.40.
- Strip sequence artifacts from 97 entries: N/C-terminal His-tags, TEV sites, GGGGS linkers, FLAG, StrepII, Factor Xa, and thrombin sites
- Drop 2 full-IgG entries (6d6u, 6d6t) -- anomalously long, not Fab/Fv fragments

Final: 491 entries. The raw CSV is cached to Drive to avoid re-downloading on subsequent runs.

Note: do not use PyTDC to access this dataset -- it is incompatible with Colab's numpy environment.

In [9]:
import urllib.request

raw_sabdab_path = DRIVE_ROOT / 'sabdab_raw.csv'

if not raw_sabdab_path.exists():
    urllib.request.urlretrieve(SABDAB_URL, str(raw_sabdab_path))
    print(f"Downloaded SAbDab to {raw_sabdab_path}")
else:
    print(f"Using cached download at {raw_sabdab_path}")

raw_sabdab_df = pd.read_csv(raw_sabdab_path)
print(f"Raw shape: {raw_sabdab_df.shape}")
print(f"Columns: {raw_sabdab_df.columns.tolist()}")

sabdab_df = parse_sabdab_raw(raw_sabdab_df)
print(f"\nCleaned: {len(sabdab_df)} rows (expected {N_SABDAB})")
print(sabdab_df[['Antibody_ID', 'pKd']].describe())

Downloaded SAbDab to /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/sabdab_raw.csv
Raw shape: (493, 5)
Columns: ['Antibody_ID', 'Antibody', 'Antigen_ID', 'Antigen', 'Y']

Cleaned: 491 rows (expected 491)
              pKd
count  491.000000
mean     8.228588
std      1.504643
min      3.698970
25%      7.253104
50%      8.204815
75%      9.148785
max     12.397940


## Mutant Sequence Reconstruction

Each row in the AbAgym data describes a single amino acid substitution: which chain (H or L), which position (PDB site label), the wildtype residue, and the mutant residue. To embed mutant sequences with ESM-2 and AbLang2, we need the full amino acid string with the substitution applied.

For each mutation: look up `seq_idx` from the ANARCI mapping for the relevant chain, replace the amino acid at that index, and verify that exactly 1 position differs from the wildtype sequence. The unchanged chain is carried over unmodified.

This produces two new columns: `mutant_heavy_seq` and `mutant_light_seq`. For a heavy chain mutation, `mutant_light_seq` is identical to the wildtype light chain, and vice versa.

In [10]:
mutant_heavy_seqs = []
mutant_light_seqs = []

for row in combined_df.itertuples(index=False):
    h_wt    = wt_sequences[row.DMS_name]['H']
    l_wt    = wt_sequences[row.DMS_name]['L']
    mapping = anarci_mappings[row.DMS_name]

    mut_h, mut_l = reconstruct_mutant_sequence(
        heavy_seq=h_wt,
        light_seq=l_wt,
        chain=row.chains,
        site=str(row.site),
        wildtype_aa=row.wildtype,
        mutant_aa=row.mutation,
        mapping=mapping,
    )
    mutant_heavy_seqs.append(mut_h)
    mutant_light_seqs.append(mut_l)

combined_df = combined_df.copy()
combined_df['mutant_heavy_seq'] = mutant_heavy_seqs
combined_df['mutant_light_seq'] = mutant_light_seqs

print(f"Mutant sequences added. Shape: {combined_df.shape}")

# Sanity: each mutant seq should differ from wildtype at exactly 1 position
sample = combined_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    chain   = row['chains']
    wt_seq  = wt_sequences[row['DMS_name']][chain]
    mut_seq = row['mutant_heavy_seq'] if chain == 'H' else row['mutant_light_seq']
    diffs   = sum(a != b for a, b in zip(wt_seq, mut_seq))
    print(f"  {row['DMS_name']} {chain}:{row['wildtype']}{row['site']}{row['mutation']} -> {diffs} diff(s)")

Mutant sequences added. Shape: (5318, 13)
  lysozyme_2019_D441 H:F64E -> 1 diff(s)
  lysozyme_2019_D441 L:C88V -> 1 diff(s)
  lysozyme_2019_D441 L:I29T -> 1 diff(s)
  EGFR_2013_Cetuximab L:I55K -> 1 diff(s)
  Ang2_2017_G6 H:G54K -> 1 diff(s)


## CDR/FR Region Labeling

Each mutation site is labeled as belonging to a CDR (complementarity-determining region) or the framework (FR). This label drives the CDR constraint loss in training.

**Why ANARCI is needed:** PDB residue numbering is arbitrary -- different structures use different numbering schemes, and insertion codes (e.g. `100A`) are a workaround for extra residues. ANARCI standardizes this by aligning the antibody sequence to a reference and assigning IMGT position numbers. Once in IMGT space, CDR/FR assignment is a fixed lookup.

**IMGT CDR boundaries** (same for both H and L chains):
- CDR1: positions 27-38
- CDR2: positions 56-65
- CDR3: positions 105-117

**HMMER dependency:** ANARCI uses HMMER (a profile hidden Markov model tool) internally for sequence alignment. HMMER must be installed at the system level before ANARCI: `apt-get install -y hmmer` on Colab, `brew install hmmer` locally.

**Canonical verification:** Ang2_2017_G6 chain H, PDB site `100A` → IMGT 113 → CDR_H3, seq_idx=104. IMGT 113 falls within the CDR_H3 range (105-117).

In [11]:
regions = []
for row in combined_df.itertuples(index=False):
    chain_map = anarci_mappings[row.DMS_name][row.chains]
    site = str(row.site)
    assert site in chain_map, (
        f"Site {site!r} not found in {row.DMS_name} chain {row.chains} mapping"
    )
    regions.append(chain_map[site]['region'])

combined_df = combined_df.copy()
combined_df['region'] = regions

print("Region distribution:")
print(combined_df['region'].value_counts())
print(f"\nUnique regions: {sorted(combined_df['region'].unique())}")

Region distribution:
region
FR        2188
CDR_H3     851
CDR_L3     646
CDR_H2     558
CDR_L1     433
CDR_H1     415
CDR_L2     227
Name: count, dtype: int64

Unique regions: ['CDR_H1', 'CDR_H2', 'CDR_H3', 'CDR_L1', 'CDR_L2', 'CDR_L3', 'FR']


## Verification

Two checks before saving.

**Structural assertions** -- confirm expected row counts, column count, and that all region labels are valid. Fails loudly if anything is wrong.

**Dataset-level breakdown** -- per-dataset CDR/FR split and overall chain distribution. Expected values based on IMGT CDR boundaries (CDR1: 27-38, CDR2: 56-65, CDR3: 105-117):

| Dataset | Total | CDR | FR |
|---|---|---|---|
| Ang2_2017_G6 | 981 | 79% | 21% |
| EGFR_2013_Cetuximab | 1071 | 65% | 35% |
| HER2_2021_trastuzumab | 184 | 100% | 0% |
| VEGF_2017b_G6 | 988 | 79% | 21% |
| lysozyme_2019_D441 | 2094 | 34% | 66% |

Chain breakdown: 3067 heavy (H), 2251 light (L). HER2 is entirely heavy chain CDR H3 mutations -- it contributes zero FR examples and zero CDR constraint gradient from the light chain.

In [12]:
expected_cols = [
    'DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation',
    'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score',
    'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance',
    'mutant_heavy_seq', 'mutant_light_seq', 'region',
]

assert len(combined_df) == N_MUTATIONS, (
    f"Expected {N_MUTATIONS} rows, got {len(combined_df)}"
)
missing = [c for c in expected_cols if c not in combined_df.columns]
assert not missing, f"Missing columns: {missing}"

assert len(sabdab_df) == N_SABDAB, (
    f"Expected {N_SABDAB} rows, got {len(sabdab_df)}"
)

print(f"combined_df: {combined_df.shape}  (expected ({N_MUTATIONS}, {len(expected_cols)}))")
print(f"sabdab_df:   {sabdab_df.shape}    (expected ({N_SABDAB}, 7))")
print("All assertions passed.")

combined_df: (5318, 14)  (expected (5318, 14))
sabdab_df:   (491, 7)    (expected (491, 7))
All assertions passed.


In [14]:
# Per-dataset CDR/FR breakdown and chain distribution
print("=== Per-dataset CDR/FR breakdown ===")
for name, grp in combined_df.groupby('DMS_name'):
    total = len(grp)
    fr    = (grp['region'] == 'FR').sum()
    cdr   = total - fr
    print(f"  {name}: {total} total | CDR={cdr} ({100*cdr/total:.0f}%) | FR={fr} ({100*fr/total:.0f}%)")

print()
print("=== Chain breakdown ===")
print(combined_df['chains'].value_counts().to_string())

=== Per-dataset CDR/FR breakdown ===
  Ang2_2017_G6: 981 total | CDR=772 (79%) | FR=209 (21%)
  EGFR_2013_Cetuximab: 1071 total | CDR=691 (65%) | FR=380 (35%)
  HER2_2021_trastuzumab: 184 total | CDR=184 (100%) | FR=0 (0%)
  VEGF_2017b_G6: 988 total | CDR=779 (79%) | FR=209 (21%)
  lysozyme_2019_D441: 2094 total | CDR=704 (34%) | FR=1390 (66%)

=== Chain breakdown ===
chains
H    3067
L    2251


## Save

Three CSVs saved to `data/` at the repo root (git-tracked):

- `abagym_antibody.csv` -- 5318 rows, 14 columns. One row per mutation. Includes reconstructed mutant sequences and CDR/FR region label.
- `abagym_sequences.csv` -- 5 rows, one per antibody. Contains wildtype heavy and light chain sequences and ANARCI mappings serialized as JSON strings. Used by embedding modules to look up `seq_idx` for residue-level extraction.
- `sabdab_affinity.csv` -- 491 rows, 7 columns. Cleaned sequences and pKd labels.

These files are committed to git so any collaborator can access them after `git clone` without re-running this pipeline.

In [13]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

# abagym_antibody.csv -- 5318 mutation rows, 14 columns
antibody_cols = [
    'DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation',
    'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score',
    'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance',
    'mutant_heavy_seq', 'mutant_light_seq', 'region',
]
combined_df[antibody_cols].to_csv(DATA_DIR / 'abagym_antibody.csv', index=False)

# abagym_sequences.csv -- 5 rows (one per antibody), ANARCI mappings as JSON strings
sequences_rows = []
for dms_name in ABAGYM_DATASETS:
    sequences_rows.append({
        'dms_name':  dms_name,
        'heavy_seq': wt_sequences[dms_name]['H'],
        'light_seq': wt_sequences[dms_name]['L'],
        'mapping_H': json.dumps(anarci_mappings[dms_name]['H']),
        'mapping_L': json.dumps(anarci_mappings[dms_name]['L']),
    })
sequences_df = pd.DataFrame(sequences_rows)
sequences_df.to_csv(DATA_DIR / 'abagym_sequences.csv', index=False)

# sabdab_affinity.csv -- 491 rows
sabdab_df.to_csv(DATA_DIR / 'sabdab_affinity.csv', index=False)

print(f"Saved to {DATA_DIR}:")
print(f"  abagym_antibody.csv   {len(combined_df)} rows x {len(antibody_cols)} cols")
print(f"  abagym_sequences.csv  {len(sequences_df)} rows x {len(sequences_df.columns)} cols")
print(f"  sabdab_affinity.csv   {len(sabdab_df)} rows x {len(sabdab_df.columns)} cols")

Saved to /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project/data:
  abagym_antibody.csv   5318 rows x 14 cols
  abagym_sequences.csv  5 rows x 5 cols
  sabdab_affinity.csv   491 rows x 7 cols
